# edge-microdot DuckDB sweep (PC)

Download `/api/export/bundle` or copy `var/snapshots/*.parquet` from the host. The same `sql/fdd/rules/*.sql` cookbooks run against `fdd.input_sample` / `fdd.point` / `sensor_history`.


In [ ]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt

SNAP_DIR = Path("../var/snapshots")
SQL_DIR = Path("../sql/fdd/rules")
con = duckdb.connect(":memory:")
con.execute("SET TimeZone = 'UTC'")
sample = sorted(SNAP_DIR.glob("sample_store_*.parquet"))[-1]
point = sorted(SNAP_DIR.glob("point_*.parquet"))[-1]
con.execute("CREATE SCHEMA IF NOT EXISTS fdd")
con.execute(f"CREATE VIEW fdd.input_sample AS SELECT site_id, point_key, ts, value FROM read_parquet('{sample}')")
con.execute(f"CREATE VIEW fdd.point AS SELECT * FROM read_parquet('{point}')")
con.execute(
    "CREATE VIEW sensor_history AS SELECT p.equipment_id, p.point_key AS series_id, "
    "p.role AS sensor_role, s.ts AS timestamp_utc, s.value, p.unit "
    "FROM fdd.input_sample s JOIN fdd.point p ON p.site_id = s.site_id AND p.point_key = s.point_key"
)


In [ ]:
stale_sql = (SQL_DIR / "sv_stale.sql").read_text()
stale = con.execute(stale_sql, [0.5, 0.5]).df()
display(stale.head(20))


In [ ]:
ts = con.execute(
    """
    SELECT date_trunc('minute', ts) AS bucket,
           avg(value) AS avg_si
    FROM fdd.input_sample
    WHERE value IS NOT NULL
    GROUP BY 1 ORDER BY 1
    """
).df()
plt.figure(figsize=(10, 3))
plt.plot(ts["bucket"], ts["avg_si"])
plt.title("Average of SI samples")
plt.tight_layout()
plt.show()
